# 02 - Modelagem genética

Neste notebook, utilizamos a amostra final definida no Notebook 1 para executar os experimentos com algoritmo genético. O objetivo e avaliar configurações do modelo de roteirizacao de forma controlada e reprodutível.

## 1. Carregamento da amostra oficial

Nesta etapa, carregamos os arquivos de paradas e veículos salvos no Notebook 1. Essa base passa a ser o insumo oficial dos experimentos do algoritmo genético.

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
SAMPLES_DIR = BASE_DIR / "dataset" / "samples"

stops_path = SAMPLES_DIR / "stops_sample_100.csv"
vehicles_path = SAMPLES_DIR / "vehicles_sample_100.csv"

stops_df = pd.read_csv(stops_path)
vehicles_df = pd.read_csv(vehicles_path)

print("Stops shape:", stops_df.shape)
print("Vehicles shape:", vehicles_df.shape)

display(stops_df.head())
display(vehicles_df)

Stops shape: (101, 7)
Vehicles shape: (4, 3)


,stop_id,name,x,y,demand,priority,is_depot
0,DEPOT-STORE,Store Depot,11.003669,76.976494,0,0,True
1,cobx423154877,médicamentos_críticos,17.501028,78.419645,2,5,False
2,gbbd486976829,médicamentos_críticos,22.340526,73.200937,2,5,False
3,mpmj126993214,médicamentos_críticos,13.023041,77.793237,2,5,False
4,opjk190252636,médicamentos_críticos,19.014049,72.845203,2,5,False


,vehicle_id,capacity,max_distance
0,motorcycle,12,25
1,van,30,40
2,scooter,10,25
3,bicycle,6,15


## 2. Validacao rapida da base de experimento

Aqui confirmamos que a amostra carregada preserva o depósito, a distribuição entre tipos de entrega e a estrutura de prioridades definida anteriormente.

In [2]:
print("Quantidade total de paradas:", len(stops_df))
print("Quantidade de entregas:", (~stops_df["is_depot"]).sum())
print("Quantidade de veículos:", len(vehicles_df))
print()

display(stops_df["name"].value_counts().to_frame("count"))
display(stops_df["priority"].value_counts().sort_index().to_frame("count"))

Quantidade total de paradas: 101
Quantidade de entregas: 100
Quantidade de veículos: 4



,count
name,
insumos_médicos,57
médicamentos_críticos,43
Store Depot,1


,count
priority,
0,1
4,22
5,78


## 3. Preparacao dos dados para o otimizador

Nesta etapa, convertemos os arquivos tabulares em estruturas de dados compatíveis com o código do projeto. Isso permite executar o algoritmo genético usando exatamente a mesma base validada no Notebook 1.

In [3]:
import sys

SRC_DIR = BASE_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from médical_route_optimizer import load_stops, load_vehicles

stops = load_stops(stops_path)
vehicles = load_vehicles(vehicles_path)

print("Quantidade de objetos Stop:", len(stops))
print("Quantidade de objetos Vehicle:", len(vehicles))
print()

print("Primeiro stop:")
print(stops[0])
print()

print("Veiculos:")
for vehicle in vehicles:
    print(vehicle)

Quantidade de objetos Stop: 101
Quantidade de objetos Vehicle: 4

Primeiro stop:
Stop(stop_id='DEPOT-STORE', name='Store Depot', x=11.003669, y=76.976494, demand=0, priority=0, is_depot=True)

Veiculos:
Vehicle(vehicle_id='motorcycle', capacity=12, max_distance=25.0)
Vehicle(vehicle_id='van', capacity=30, max_distance=40.0)
Vehicle(vehicle_id='scooter', capacity=10, max_distance=25.0)
Vehicle(vehicle_id='bicycle', capacity=6, max_distance=15.0)


## 4. Definicao do experimento base

Antes de comparar configurações, definimos um experimento inicial para validar o fluxo completo do algoritmo genético sobre a amostra selecionada. Esse experimento servira como baseline para as variacoes posteriores.

In [6]:
ga_config_base = {
    "population_size": 80,
    "generations": 120,
    "mutation_rate": 0.15,
    "elite_size": 2,
    "random_seed": 42,
}

ga_config_base

{'population_size': 80,
 'generations': 120,
 'mutation_rate': 0.15,
 'elite_size': 2,
 'random_seed': 42}

## 5. Execucao do experimento base

Nesta etapa, executamos a configuração inicial do algoritmo genético sobre a amostra oficial. O objetivo e validar o comportamento do modelo, medir o fitness final e observar a distribuição das entregas entre os veículos.

In [7]:
from médical_route_optimizer import optimize_routes

result_base = optimize_routes(
    stops=stops,
    vehicles=vehicles,
    population_size=ga_config_base["population_size"],
    generations=ga_config_base["generations"],
    mutation_rate=ga_config_base["mutation_rate"],
    elite_size=ga_config_base["elite_size"],
    random_seed=ga_config_base["random_seed"],
)

print("Best fitness:", result_base.best_fitness)
print("Total distance:", result_base.total_distance)
print("Historico de geracoes:", len(result_base.history))

Best fitness: 53739.59
Total distance: 382.69
Historico de geracoes: 120


## 6. Leitura inicial do resultado

Após a execucao do baseline, analisamos o custo total, o numero de geracoes registradas e a distribuição das rotas por veículo. Essa leitura inicial ajuda a identificar gargalos antes de comparar novas configurações.

In [8]:
route_summary = pd.DataFrame(
    [
        {
            "vehicle_id": route.vehicle_id,
            "num_stops": len(route.stop_ids),
            "distance": route.distance,
            "load": route.load,
            "capacity_overflow": route.capacity_overflow,
            "distance_overflow": route.distance_overflow,
        }
        for route in result_base.routes
    ]
)

display(route_summary)

,vehicle_id,num_stops,distance,load,capacity_overflow,distance_overflow
0,motorcycle,24,80.24,63,51,55.24
1,van,28,110.68,70,40,70.68
2,scooter,25,90.06,64,54,65.06
3,bicycle,23,101.71,60,54,86.71


## 7. Diagnostico de viabilidade da configuração base

O primeiro experimento mostrou estouro generalizado de capacidade e distância. Antes de testar novas configurações do algoritmo genético, precisamos verificar se a amostra e os limites dos veículos sao compativeis entre si.

In [9]:
total_demand = stops_df.loc[~stops_df["is_depot"], "demand"].sum()
total_capacity = vehicles_df["capacity"].sum()

print("Demanda total da amostra:", total_demand)
print("Capacidade total disponível:", total_capacity)
print("Razao demanda/capacidade:", round(total_demand / total_capacity, 2))
print()

display(vehicles_df)

Demanda total da amostra: 257
Capacidade total disponível: 58
Razao demanda/capacidade: 4.43



,vehicle_id,capacity,max_distance
0,motorcycle,12,25
1,van,30,40
2,scooter,10,25
3,bicycle,6,15


## 8. Diagnostico da autonomia total

Além da capacidade de carga, também verificamos se a autonomia total da frota e compatível com a extensao das rotas produzidas. Essa leitura ajuda a separar um problema de parametrizacao da frota de um problema do algoritmo em si.

In [10]:
distance_capacity_total = vehicles_df["max_distance"].sum()

print("Distancia máxima total disponível:", distance_capacity_total)
print("Distancia total encontrada no baseline:", result_base.total_distance)
print("Razao distância usada/capacidade total:", round(result_base.total_distance / distance_capacity_total, 2))

Distancia máxima total disponível: 105
Distancia total encontrada no baseline: 382.69
Razao distância usada/capacidade total: 3.64


## 9. Recalibracao inicial da frota

A primeira calibracao amplia a capacidade e a autonomia da frota para verificar se a instancia deixa de ser inviavel em termos absolutos. O objetivo desta etapa nao e chegar a configuração final, mas reduzir os estouros mais graves antes de uma segunda leitura do baseline.

In [11]:
vehicles_experiment_df = vehicles_df.copy()

vehicles_experiment_df["capacity"] = [45, 70, 45, 30]
vehicles_experiment_df["max_distance"] = [120, 180, 120, 90]

display(vehicles_experiment_df)

print("Nova capacidade total:", vehicles_experiment_df["capacity"].sum())
print("Nova distância máxima total:", vehicles_experiment_df["max_distance"].sum())

,vehicle_id,capacity,max_distance
0,motorcycle,45,120
1,van,70,180
2,scooter,45,120
3,bicycle,30,90


Nova capacidade total: 190
Nova distância máxima total: 510


## 10. Reconstrucao da frota recalibrada

Depois do primeiro ajuste, salvamos a frota recalibrada e recriamos os objetos utilizados pelo otimizador para medir o efeito dessa mudanca sobre o baseline.

In [12]:
vehicles_experiment_path = SAMPLES_DIR / "vehicles_sample_100_experiment.csv"
vehicles_experiment_df.to_csv(vehicles_experiment_path, index=False)

vehicles_experiment = load_vehicles(vehicles_experiment_path)

for vehicle in vehicles_experiment:
    print(vehicle)

Vehicle(vehicle_id='motorcycle', capacity=45, max_distance=120.0)
Vehicle(vehicle_id='van', capacity=70, max_distance=180.0)
Vehicle(vehicle_id='scooter', capacity=45, max_distance=120.0)
Vehicle(vehicle_id='bicycle', capacity=30, max_distance=90.0)


## 11. Segunda calibracao da frota

A primeira recalibracao reduziu os estouros de distância, mas a distribuição de carga ainda permaneceu inviavel. Como a amostra de 100 entregas será mantida, foi necessario realizar um segundo ajuste nas capacidades da frota para tornar a instancia adequada aos experimentos comparativos.

In [15]:
vehicles_experiment_df = vehicles_df.copy()

vehicles_experiment_df["capacity"] = [65, 85, 65, 45]
vehicles_experiment_df["max_distance"] = [120, 180, 120, 90]

display(vehicles_experiment_df)

print("Nova capacidade total:", vehicles_experiment_df["capacity"].sum())
print("Nova distância máxima total:", vehicles_experiment_df["max_distance"].sum())

,vehicle_id,capacity,max_distance
0,motorcycle,65,120
1,van,85,180
2,scooter,65,120
3,bicycle,45,90


Nova capacidade total: 260
Nova distância máxima total: 510


## 12. Atualizacao da frota de experimento

Após o novo ajuste de capacidade, salvamos a versao atualizada da frota e recriamos os objetos utilizados pelo otimizador. Isso garante que a próxima execucao do baseline utilize os parâmetros recalibrados de forma consistente.

In [25]:
vehicles_experiment_df.to_csv(vehicles_experiment_path, index=False)
vehicles_experiment = load_vehicles(vehicles_experiment_path)

for vehicle in vehicles_experiment:
    print(vehicle)

Vehicle(vehicle_id='motorcycle', capacity=65, max_distance=140.0)
Vehicle(vehicle_id='van', capacity=85, max_distance=210.0)
Vehicle(vehicle_id='scooter', capacity=65, max_distance=140.0)
Vehicle(vehicle_id='bicycle', capacity=45, max_distance=120.0)


## 13. Ajuste final de autonomia

A segunda calibracao resolveu os estouros de capacidade, mas a autonomia da frota ainda permaneceu insuficiente para parte das rotas. Como o objetivo desta etapa e obter uma instancia viavel para comparação entre configurações do algoritmo genético, foi realizado um ajuste final apenas nos limites de distância.

In [19]:
vehicles_experiment_df = vehicles_df.copy()

vehicles_experiment_df["capacity"] = [65, 85, 65, 45]
vehicles_experiment_df["max_distance"] = [140, 210, 140, 120]

display(vehicles_experiment_df)

print("Capacidade total:", vehicles_experiment_df["capacity"].sum())
print("Distancia máxima total:", vehicles_experiment_df["max_distance"].sum())

,vehicle_id,capacity,max_distance
0,motorcycle,65,140
1,van,85,210
2,scooter,65,140
3,bicycle,45,120


Capacidade total: 260
Distancia máxima total: 610


## 14. Aplicacao da frota final de experimento

Com os limites finais de capacidade e autonomia definidos, salvamos a frota recalibrada e recriamos os objetos utilizados pelo otimizador para a nova execucao do baseline.

In [20]:
vehicles_experiment_df.to_csv(vehicles_experiment_path, index=False)
vehicles_experiment = load_vehicles(vehicles_experiment_path)

for vehicle in vehicles_experiment:
    print(vehicle)

Vehicle(vehicle_id='motorcycle', capacity=65, max_distance=140.0)
Vehicle(vehicle_id='van', capacity=85, max_distance=210.0)
Vehicle(vehicle_id='scooter', capacity=65, max_distance=140.0)
Vehicle(vehicle_id='bicycle', capacity=45, max_distance=120.0)


## 15. Reexecucao do baseline após recalibracao da frota

Depois dos ajustes de capacidade e autonomia, reexecutamos o experimento base para verificar se a instancia passou a ser operacionalmente viavel. O objetivo desta etapa e validar se os limites da frota ficaram compatíveis com a amostra de entregas antes de iniciar a comparação entre diferentes configurações do algoritmo genético.

In [21]:
result_base_recalibrado = optimize_routes(
    stops=stops,
    vehicles=vehicles_experiment,
    population_size=ga_config_base["population_size"],
    generations=ga_config_base["generations"],
    mutation_rate=ga_config_base["mutation_rate"],
    elite_size=ga_config_base["elite_size"],
    random_seed=ga_config_base["random_seed"],
)

print("Best fitness:", result_base_recalibrado.best_fitness)
print("Total distance:", result_base_recalibrado.total_distance)
print("Historico de geracoes:", len(result_base_recalibrado.history))

Best fitness: 31519.4
Total distance: 575.4
Historico de geracoes: 120


## 16. Reavaliação das rotas geradas

Nesta etapa, analisamos novamente a distribuição das rotas por veículo, com foco nos indicadores de carga, distância percorrida e eventuais estouros residuais. A partir dessa leitura, decidimos se a calibracao da frota foi suficiente para sustentar os próximos experimentos.

In [22]:
route_summary_recalibrado = pd.DataFrame(
    [
        {
            "vehicle_id": route.vehicle_id,
            "num_stops": len(route.stop_ids),
            "distance": route.distance,
            "load": route.load,
            "capacity_overflow": route.capacity_overflow,
            "distance_overflow": route.distance_overflow,
        }
        for route in result_base_recalibrado.routes
    ]
)

display(route_summary_recalibrado)

,vehicle_id,num_stops,distance,load,capacity_overflow,distance_overflow
0,motorcycle,24,140.90,65,0,0.9
1,van,31,204.71,84,0,0.0
2,scooter,25,133.35,64,0,0.0
3,bicycle,20,96.44,44,0,0.0


## 17. Conclusao da calibracao

A calibracao final da frota eliminou os estouros de capacidade e reduziu os excessos de distância a um nível residual. Com isso, a instancia passou a ser adequada para a comparação entre diferentes configurações do algoritmo genético.

Dessa forma, a frota recalibrada será mantida como baseline operacional dos experimentos a seguir.

## 18. Definicao dos experimentos

Com a instancia calibrada, definimos tres configurações do algoritmo genético para avaliar o impacto de parâmetros como tamanho da populacao, numero de geracoes e taxa de mutacao. O objetivo e comparar estabilidade, custo total e qualidade das rotas geradas.

In [23]:
experiment_configs = {
    "exp_1_base": {
        "population_size": 80,
        "generations": 120,
        "mutation_rate": 0.15,
        "elite_size": 2,
        "random_seed": 42,
    },
    "exp_2_more_generations": {
        "population_size": 80,
        "generations": 200,
        "mutation_rate": 0.15,
        "elite_size": 2,
        "random_seed": 42,
    },
    "exp_3_more_population": {
        "population_size": 120,
        "generations": 120,
        "mutation_rate": 0.10,
        "elite_size": 2,
        "random_seed": 42,
    },
}

experiment_configs

{'exp_1_base': {'population_size': 80,
  'generations': 120,
  'mutation_rate': 0.15,
  'elite_size': 2,
  'random_seed': 42},
 'exp_2_more_generations': {'population_size': 80,
  'generations': 200,
  'mutation_rate': 0.15,
  'elite_size': 2,
  'random_seed': 42},
 'exp_3_more_population': {'population_size': 120,
  'generations': 120,
  'mutation_rate': 0.1,
  'elite_size': 2,
  'random_seed': 42}}

## 19. Execucao dos experimentos

Nesta etapa, executamos as tres configurações definidas anteriormente sobre a mesma amostra e a mesma frota recalibrada. Isso garante comparabilidade direta entre os resultados.

In [26]:
experiment_results = {}

for experiment_name, config in experiment_configs.items():
    result = optimize_routes(
        stops=stops,
        vehicles=vehicles_experiment,
        population_size=config["population_size"],
        generations=config["generations"],
        mutation_rate=config["mutation_rate"],
        elite_size=config["elite_size"],
        random_seed=config["random_seed"],
    )
    experiment_results[experiment_name] = result
    print(
        experiment_name,
        "| fitness =", result.best_fitness,
        "| total_distance =", result.total_distance,
        "| generations =", len(result.history),
    )

exp_1_base | fitness = 31519.4 | total_distance = 575.4 | generations = 120
exp_2_more_generations | fitness = 31419.74 | total_distance = 580.94 | generations = 200
exp_3_more_population | fitness = 31403.7 | total_distance = 583.7 | generations = 120


## 20. Comparacao entre os experimentos

Após executar as tres configurações, consolidamos os resultados em uma tabela comparativa para analisar o impacto dos parâmetros do algoritmo genético sobre o fitness final, a distância total e a estabilidade da busca.

In [27]:
comparison_df = pd.DataFrame(
    [
        {
            "experiment": experiment_name,
            "population_size": experiment_configs[experiment_name]["population_size"],
            "generations": experiment_configs[experiment_name]["generations"],
            "mutation_rate": experiment_configs[experiment_name]["mutation_rate"],
            "elite_size": experiment_configs[experiment_name]["elite_size"],
            "best_fitness": result.best_fitness,
            "total_distance": result.total_distance,
            "history_length": len(result.history),
        }
        for experiment_name, result in experiment_results.items()
    ]
).sort_values("best_fitness")

display(comparison_df)

,experiment,population_size,generations,mutation_rate,elite_size,best_fitness,total_distance,history_length
2,exp_3_more_population,120,120,0.10,2,31403.70,583.70,120
1,exp_2_more_generations,80,200,0.15,2,31419.74,580.94,200
0,exp_1_base,80,120,0.15,2,31519.40,575.40,120


## 21. Interpretacao inicial dos resultados

Nesta etapa, observamos qual configuração apresentou o menor fitness e como isso se refletiu na distância total. Como o problema ainda incorpora penalidades operacionais, a comparação deve priorizar o fitness como medida principal de qualidade da solução, sem ignorar o comportamento da distância total.

In [28]:
best_experiment_row = comparison_df.iloc[0]

print("Melhor experimento:", best_experiment_row["experiment"])
print("Best fitness:", best_experiment_row["best_fitness"])
print("Total distance:", best_experiment_row["total_distance"])

Melhor experimento: exp_3_more_population
Best fitness: 31403.7
Total distance: 583.7


## 22. Sintese da etapa

A etapa de modelagem genética mostrou que a base amostrada exigiu recalibracao da frota para se tornar operacionalmente viavel. Após os ajustes de capacidade e autonomia, foi possível executar e comparar tres configurações do algoritmo genético sobre a mesma instancia.

Entre os experimentos realizados, a configuração `exp_3_more_population` apresentou o menor fitness final, indicando melhor desempenho global segundo a função objetivo adotada. Embora sua distância total nao tenha sido a menor entre os tres cenarios, o resultado sugere que o aumento do tamanho da populacao produziu solucoes mais competitivas do que apenas ampliar o numero de geracoes.

Com isso, a configuração `exp_3_more_population` será adotada como referencia para o Notebook 3, dedicado a avaliação final e a visualizacao das rotas geradas.